In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import collections
from pathlib import Path
from scipy.signal import savgol_filter

import sys
sys.path.append('../../src/apriltag')
from read_tags import Detection as MyTag

In [ ]:
folder = Path('/Users/gerry/DropboxGatech/Painting/Graffiti_media+data/2024-02-03_iros2024_homography_data/data/')

## Load Log

In [ ]:
# Load a log
def parse_line(line):
    # Splitting the line by spaces and filtering out empty strings
    # Example line:
    #   0       1 2  3      4      5      6 7      8      9
    #   4191717 - 0: 0.0000 0.7473 1.5038 - 0.0000 0.7500 1.5000 | 0 8 1.2284 0.3533 | 0 8 0.4701 0.2876 | 0 8 -1.2426 0.8971 | 0 8 -0.2129 0.9118 | 1
    parts = [part for part in line.split() if part.strip()]
    return int(parts[0]) / 1e6, float(parts[4]), float(parts[5]), float(parts[8]), float(parts[9])

def parse_file(file):
    with open(file, 'r') as f:
        lines = f.readlines()
    data = np.array([parse_line(line) for line in lines])
    t = data[:, 0]
    xy_est = data[:, 1:3]
    xy_com = data[:, 3:5]
    return t, xy_est, xy_com

line = '4191717 - 0: 0.0000 0.7473 1.5038 - 0.0000 0.7500 1.5000 | 0 8 1.2284 0.3533 | 0 8 0.4701 0.2876 | 0 8 -1.2426 0.8971 | 0 8 -0.2129 0.9118 | 1'
parsed_line = parse_line(line)
assert parsed_line == (4.191717, 0.7473, 1.5038, 0.75, 1.5), parsed_line

t, xy_est, xy_com = parse_file(folder / 'A.log')
print(t.shape, xy_est.shape, xy_com.shape)
for i in range(3):
    print(t[i] / 1e6, xy_est[i], xy_com[i])

In [ ]:
plt.plot(xy_est[:, 0], xy_est[:, 1], label='Estimated')
plt.plot(xy_com[:, 0], xy_com[:, 1], label='Commanded')

## Load AprilTag Detections

Expect:
* 1: BR
* 2: TR
* 3: TL
* 4: BL
* 9: End Effector

Tags themselves were printed 3.75" (50% scale at 7.5"), but that's black to black.  I think it might be off.

Camera was about 175" away from the tags.  
Tags are inset by paper-center away from the corner of the frame, which in turn is 4" vertically beyond the top/bottom corners (but horizontally in line).  
End effector tag is inset from BL origin corner by about x: 0.75" y: 0.625"

In [ ]:
inch2m = lambda x: x * 0.0254
m2inch = lambda x: x / 0.0254
W, H = 2.9464, 2.26
w_paper, h_paper = inch2m(8.5), inch2m(11)
w_tag, h_tag = inch2m(3.75), inch2m(3.75)
w_between_tags, h_between_tags = (W - w_paper), (H - h_paper + inch2m(4)*2)
print(m2inch(w_between_tags), m2inch(h_between_tags))
x_offset, y_offset = (w_paper - w_tag) / 2 - inch2m(0.75), (h_paper - h_tag) / 2 - inch2m(4) - inch2m(0.625)
print(m2inch(x_offset), m2inch(y_offset))

In [ ]:
with np.load(folder / 'camera_calibration.npz') as data:
    K = data['mtx']
    dist = data['dist']
    rvecs = data['rvecs']
    tvecs = data['tvecs']
    print(K, dist, rvecs.shape, tvecs.shape)

In [ ]:
with np.load(folder / 'A_detections_every10.npz', allow_pickle=True) as f:
    all_detections = f['all_detections'].item()
assert all(len(detections) == 5 for detections in all_detections.values()), "Not all video frames have 5 tags"

detections = all_detections[0]
print(len(all_detections), len(detections))
print()
print(detections[0])

plt.figure(figsize=(4, 3))
ts = []
for detection in detections:
    tag = MyTag(corners=detection.corners)
    R, t = tag.Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)  # divide by 2 due to idk why
    ts.append(t)
    plt.plot(t[0], t[1], 'o')
    plt.text(t[0], t[1], f'{detection.tag_id}', fontsize=8)
    with np.printoptions(precision=3, suppress=True):
        print(R)
        print(t)
        print()


In [ ]:
with np.printoptions(precision=3, suppress=True):
    print('height between apriltags', np.linalg.norm(ts[0] - ts[1]))
    print('width  between apriltags', np.linalg.norm(ts[1] - ts[2]))
    print('height between apriltags', np.linalg.norm(ts[2] - ts[3]))
    print('width  between apriltags', np.linalg.norm(ts[3] - ts[0]))
print('expected width between apriltags:', m2inch(w_between_tags))
print('expected height between apriltags:', m2inch(h_between_tags))
scale_factor = np.mean([np.linalg.norm(ts[0] - ts[1]) / m2inch(h_between_tags),
                        np.linalg.norm(ts[1] - ts[2]) / m2inch(w_between_tags),
                        np.linalg.norm(ts[2] - ts[3]) / m2inch(h_between_tags),
                        np.linalg.norm(ts[3] - ts[0]) / m2inch(w_between_tags)])
print('scale factor:', scale_factor)
print('uncorrected height between apriltags:', m2inch(h_between_tags) * scale_factor)
print('uncorrected width  between apriltags:', m2inch(w_between_tags) * scale_factor)

In [ ]:
for detection in detections:
    print(detection.tag_id, detection.corners[0])

In [ ]:
# Compute an xy homography correction using the 4 corners
def compute_H(act, est):
    # act is 2x4, est is 2x4
    A = np.zeros((8, 9))
    for i in range(4):
        A[2*i, 0:3] = act[:, i]
        A[2*i, 6:9] = -est[0, i] * act[:, i]
        A[2*i+1, 3:6] = act[:, i]
        A[2*i+1, 6:9] = -est[1, i] * act[:, i]
    _, _, V = np.linalg.svd(A)
    H = V[-1, :].reshape(3, 3)
    return H

def make_homogeneous(x):
    return np.vstack((x, np.ones(x.shape[1])))

def apply_H(H, good):
    pred = H @ make_homogeneous(good.T)
    pred /= pred[-1, :]
    return pred[:-1, :].T

In [ ]:
all_trajs = {}
for trial in 'ABCD':
    with np.load(folder / f'{trial}_detections_every1.npz', allow_pickle=True) as f:
        all_detections = f['all_detections'].item()
    old_len = len(all_detections)
    all_detections = {k: v for k, v in all_detections.items() if len(v) == 5}
    assert len(all_detections) + 10 >= old_len, "Not all video frames have 5 tags"

    def get_pos(detections):
        detections_ = {det.tag_id: det for det in detections}
        _, ee = MyTag(corners = detections_[9].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        _, bl = MyTag(corners = detections_[4].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        return np.array([ee[0] - bl[0], -(ee[1] - bl[1])]) / scale_factor
    
    def get_pos2(detections):
        detections_ = {det.tag_id: det for det in detections}
        exp = np.array([[w_between_tags, 0], [w_between_tags, h_between_tags], [0, h_between_tags], [0, 0]])
        act = np.zeros((3, 4))
        _, act[:, 0:1] = MyTag(corners = detections_[1].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        _, act[:, 1:2] = MyTag(corners = detections_[2].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        _, act[:, 2:3] = MyTag(corners = detections_[3].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        _, act[:, 3:4] = MyTag(corners = detections_[4].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        _, ee = MyTag(corners = detections_[9].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K, dist_coeffs=dist)
        H = compute_H(make_homogeneous(act[:2, :]), exp.T)
        # print(act)
        # print(exp)
        # print(ee)
        # print(apply_H(H, ee[:2].T))
        # raise Exception()
        return m2inch(apply_H(H, ee[:2].T))

    traj = np.array([get_pos2(detections) for _, detections in sorted(all_detections.items())]).squeeze()
    traj += np.array([m2inch(x_offset), m2inch(y_offset)])
    ts = np.array([k / 60 for k in sorted(all_detections.keys())])

    all_trajs[trial] = (ts, inch2m(traj))
all_trajs_orig = {k: v for k, v in all_trajs.items()}

In [ ]:
correct = np.load('data/zigzag.npy')

In [ ]:
print(correct.shape, all_trajs['A'][0].shape)

In [ ]:
# remove outliers
all_trajs2 = {}
for k, (ts, traj) in all_trajs_orig.items():
    xsmooth = savgol_filter(traj[:, 0], 11, 3)
    ysmooth = savgol_filter(traj[:, 1], 11, 3)
    xysmooth = np.column_stack((xsmooth, ysmooth))
    bad = np.linalg.norm(xysmooth - traj, axis=1) > 0.01
    all_trajs2[k] = (ts[~bad], traj[~bad])
all_trajs = all_trajs2

In [ ]:
# smooth
all_trajs2 = {}
for k, (ts, traj) in all_trajs.items():
    xsmooth = savgol_filter(traj[:, 0], 3, 1)
    ysmooth = savgol_filter(traj[:, 1], 3, 1)
    xysmooth = np.column_stack((xsmooth, ysmooth))
    all_trajs2[k] = (ts, xysmooth)
all_trajs = all_trajs2

In [ ]:
# Filter start & stop times
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
last_is_true = lambda x: x.shape[0] - np.argmax(x[::-1]) - 1
all_trajs_retimed = {}
for k, (ts, traj) in all_trajs.items():
    ts = ts.copy()
    traj = traj.copy()

    dxy = np.diff(traj, axis=0)
    dxy = np.array([savgol_filter(dxy_, 31, 3) for dxy_ in dxy.T]).T
    speed = np.linalg.norm(dxy, axis=1) / np.diff(ts)

    start = np.argmax(speed < 0.05)  # cut out spurious noise at beginning
    end = last_is_true(speed < 0.05)  # cut out spurious noise at end
    ts, traj = ts[start:end], traj[start:end]

    dxy = np.diff(traj, axis=0)
    dxy = np.array([savgol_filter(dxy_, 31, 3) for dxy_ in dxy.T]).T
    speed = np.linalg.norm(dxy, axis=1) / np.diff(ts)

    start = np.argmax(speed > 0.1)
    stop = last_is_true(speed > 0.1)
    axes[0].plot(speed[start:stop])

    # normalize time to be from 0 - 1
    ts -= ts[start]
    ts /= ts[stop]
    all_trajs_retimed[k] = (ts[start:stop], traj[start:stop])

for k, (ts, traj) in all_trajs_retimed.items():
    axes[1].plot(ts, traj[:, 0], label=k)
    axes[2].plot(*traj.T, label=k)
axes[1].plot(np.linspace(0, 1, correct.shape[0]), correct[:, 0], label='correct')
axes[1].legend()

In [ ]:
# compute error by interpolating time
errs_timesynced = {}
for k, (ts, traj) in all_trajs_retimed.items():
    traj_interp = np.array([np.interp(ts, np.linspace(0, 1, correct.shape[0]), correct[:, 0]),
                            np.interp(ts, np.linspace(0, 1, correct.shape[0]), correct[:, 1])]).T
    error = np.linalg.norm(traj_interp - traj, axis=1)
    # plt.plot(error, label=k)
    plt.plot(traj_interp - traj, label=k)
    print(k, np.mean(error), np.std(error))
    errs_timesynced[k] = np.mean(error), np.std(error)
plt.legend()

In [ ]:
correct.shape[0]

In [ ]:
# Compute error by finding the closest point
correct_dense = np.array([np.interp(np.linspace(0, 1, 85000), np.linspace(0, 1, correct.shape[0]), correct_) for correct_ in correct.T]).T
errs_closest = {}
for k, (ts, traj) in all_trajs_retimed.items():
    errors = np.linalg.norm(traj[:, None] - correct_dense[None], axis=2)
    print(errors.shape)
    errors = np.min(errors, axis=1)
    print(k, np.mean(errors), np.std(errors))
    errs_closest[k] = np.mean(errors), np.std(errors)

In [ ]:
print(errs_timesynced)
print(errs_closest)
errs_overall = {}
for k, (mean, std) in errs_closest.items():
    if std > 0.02:
        errs_overall[k] = errs_timesynced[k]
    else:
        errs_overall[k] = errs_closest[k]

In [ ]:
fig = plt.figure(figsize=(6.40, 4.80 / 0.8))
ax = plt.subplot(111)

name_map = dict(A='Proprioceptive-only',
                B='Joint Optimization',
                C='Proprioceptive + Single Homography',
                D='Proprioceptive + Piecewise Homography (ours)')
ls = dict(A='r', B='m', C='g', D='b')
order=['A','C','D','B']
plt.plot(correct[:, 0], correct[:, 1], 'k-', label='Target Trajectory')
# for trial, (ts, traj) in all_trajs.items():
for trial in order:
    (ts, traj) = all_trajs[trial]
    print(traj.shape)
    err_label = f'({errs_overall[trial][0]*1000:3.0f}mm ATE) '
    plt.plot(traj[:, 0], traj[:, 1], ls[trial], label=err_label + name_map[trial], markersize=2, linewidth=0.5)
    # plt.plot(traj[:, 0], traj[:, 1], ls[trial].replace('.', '-'), label=name_map[trial], width=1)
    plt.plot(traj[::90, 0], traj[::90, 1], ls[trial] + 'o', markersize=3)
# plt.legend(location='')
plt.title('Trajectory Tracking Accuracy for Different Calibration Methods')
plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.axis('equal')

# Shrink current axis by 20%
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height * 0.8])
# Put a legend to the right of the current axis
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1))

plt.savefig('results/trajectory_tracking_accuracy.eps', bbox_inches='tight')
plt.savefig('results/trajectory_tracking_accuracy.png', bbox_inches='tight')